In [22]:
import cupy as np
from cupyx.scipy import signal
np.random.seed(42)

In [23]:
class Layer:
    def __init__(self):
        self.input = None
        self.output = None
    
    # calculates the output of the layer based on the input
    def forward_pass(self, input):
        pass
    
    # update the layer's parameters
    def backward_pass(self, output_gradient, learning_rate):
        pass

valid cross correlation (\star symbol): the kernel starts moving on the matrix with the edges lined up
full cross correlation (* symbol): the kernel starts moving on the matrix from the first possible intersection

convolutional layer's forward pass:
$$Y_i = B_i + \sum_{j=1}^{n} X_j \star K_{ij}$$

convolutional layer's backward pass:
output gradient (= derivative of the error with respect to the biases):
$$\frac{\partial E}{\partial B_i} = \frac{\partial E}{\partial Y_i}$$

input gradient (= derivative of the error with respect to the input):
$$\frac{\partial E}{\partial X_j} = \sum_{i=1}^{n} \frac{\partial E}{\partial Y_i} * K_{ij}$$

kernel's gradient  (= derivative of the error with respect to the kernels):
$$\frac{\partial E}{\partial K_{ij}} = X_j \star \frac{\partial E}{\partial Y_i}$$

In [24]:
class Convolutional(Layer):
    # input shape: tuple, containing depth, height and width of the input
    # kernel size: int, specifying the size of the matrix inside each kernel (it's a square matrix)
    # depth: int, specifying how many kernels there are
    def __init__(self, input_shape: tuple, kernel_size: int, depth: int):
        input_depth, input_height, input_width = input_shape
        self.depth = depth
        self.input_shape = input_shape
        self.input_depth = input_depth
        # stride = 1, no padding
        self.output_shape = (depth,
                             input_height - kernel_size + 1,
                             input_width - kernel_size + 1)
        # 4D: multiple 3D kernels
        # depth = number of kernels
        # input_depth = depth of each kernel (depth of input)
        # the last two are to create the square matrices in each kernel
        self.kernels_shape = (depth, input_depth, kernel_size, kernel_size)
        # initializing randomly
        self.kernels = np.random.randn(*self.kernels_shape)
        self.biases = np.random.randn(*self.output_shape)

    def forward_pass(self, input):
        self.input = input
        # calculating the output (implementing the above formula)
        self.output = np.copy(self.biases)
        for i in range(self.depth):
            for j in range(self.input_depth):
                # using cross correlation from scipy
                self.output[i] += signal.correlate2d(self.input[j], self.kernels[i, j], "valid")
        return self.output

    def backward_pass(self, output_gradient, learning_rate):
        # initializing the gradients with empty matrices
        kernels_gradient = np.zeros(self.kernels_shape)
        input_gradient = np.zeros(self.input_shape)
        # gradient of the biases: how the loss function changes with respect to the biases (derivative of the error with respect to the biases)
        # the bias gradient is already given as the output_gradient parameter

        for i in range(self.depth):
            for j in range(self.input_depth):
                # gradient of the input: how the loss function changes with respect to the input to the layer
                # sensitivity of the loss with respect to changes in the input values
                input_gradient[j] += signal.convolve2d(output_gradient[i], self.kernels[i, j], "full")
                # gradient of a kernel: how the loss function changes with respect to the individual elements of the kernel
                # sensitivity of the loss with respect to changes in the values of the kernel weights
                kernels_gradient[i, j] = signal.correlate2d(self.input[j], output_gradient[i], "valid")

        # updating the parameters with gradient descent
        self.kernels -= learning_rate * kernels_gradient
        self.biases -= learning_rate * output_gradient
        
        return input_gradient

dense layer's forward pass:
$$Y = W \cdot X + B$$

dense layer's backward pass:
output gradient:
$$\frac{\partial E}{\partial B} = \frac{\partial E}{\partial Y}$$

weights gradient:
$$\frac{\partial E}{\partial W} = \frac{\partial E}{\partial Y} \cdot X^t$$

input gradient:
$$\frac{\partial E}{\partial X} = W^t \cdot \frac{\partial E}{\partial Y}$$

In [25]:
class Dense(Layer):
    # n_input: number of neurons in the input
    # n_output: number of neurons in the output
    def __init__(self, n_input: int, n_output: int):
        # initializing randomly
        self.weights = np.random.randn(n_output, n_input)
        self.bias = np.random.randn(n_output, 1)

    def forward_pass(self, input):
        self.input = input
        # calculating the output (implementing the above formula)
        return np.dot(self.weights, self.input) + self.bias

    def backward_pass(self, output_gradient, learning_rate):
        # gradient of a kernel: derivative of the error with respect to the weights
        weights_gradient = np.dot(output_gradient, self.input.T)
        # gradient of the input: derivative of the error with respect to the input to the layer
        input_gradient = np.dot(self.weights.T, output_gradient)
        # gradient of the biases: derivative of the error with respect to the biases = output_gradient
        
        # updating the parameters with gradient descent
        self.weights -= learning_rate * weights_gradient
        self.bias -= learning_rate * output_gradient
        
        return input_gradient

In [26]:
class Reshape(Layer):
    def __init__(self, input_shape, output_shape):
        self.input_shape = input_shape
        self.output_shape = output_shape

    def forward_pass(self, input):
        return np.reshape(input, self.output_shape)

    def backward_pass(self, output_gradient, learning_rate):
        # in this case the learning_rate parameter isn't used, it's only there bc this is inherits from the layer class o.O
        return np.reshape(output_gradient, self.input_shape)

binary cross entropy:
$$E = {-\frac{1}{n}} \sum_{i=1}^{n}  y^*_i log(y_i)  +  (1-y^*_i)log(1-y_i)$$

y*: actual output
y: predicted output

binary cross entropy derivative:
$$\frac{\partial E}{\partial y_i} = \frac{1}{n} (\frac{1-y^*_i}{1-y_i}-\frac{y^*_i}{y_i}) $$

In [27]:
def binary_cross_entropy(y_true, y_pred):
    return np.mean(-y_true * np.log(y_pred) - (1 - y_true) * np.log(1 - y_pred))

def binary_cross_entropy_derivative(y_true, y_pred):
    return ((1 - y_true) / (1 - y_pred) - y_true / y_pred) / np.size(y_true)

sigmoid:
$$\sigma(x)=\frac{1}{1+e^{-x}}$$

sigmoid derivative (backward pass):
$$\sigma'(x)=\sigma(x)*(1-\sigma(x))$$

In [28]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

class Sigmoid(Layer):
    def __init__(self):
        pass

    def forward_pass(self, input):
        self.input = input
        return sigmoid(self.input)
    
    def backward_pass(self, output_gradient, learning_rate):
        sigma_x = sigmoid(self.input)
        return np.multiply(output_gradient, (sigma_x * (1 - sigma_x)))

relu:
$$R(x) = max(0, x)$$

relu derivative (backward pass):
R'(x) = 0 if x < 0 else 1

In [29]:
def relu(x):
    return np.maximum(0, x)

class Relu(Layer):
    def __init__(self):
        pass

    def forward_pass(self, input):
        self.input = input
        return relu(self.input)

    def backward_pass(self, output_gradient, learning_rate):
        # elements are 0 where the condition is true, and 1 where the condition is false
        relu_derivative = np.where(self.input < 0, 0, 1)
        return np.multiply(output_gradient, relu_derivative)

In [30]:
class Network:
    def __init__(self, layers: list):
        self.layers = layers
        
    def predict(self, input):
        output = input
        for layer in self.layers:
            output = layer.forward_pass(output)
        return output
    
    def train(self, loss_function, loss_function_derivative, x_train, y_train, epochs = 5, learning_rate = 0.01):
        for i in range(epochs):
            error = 0
            correct_predictions = 0

            for x, y in zip(x_train, y_train):
                # forward
                output = self.predict(x)
    
                # error
                error += loss_function(y, output)
    
                # backward
                gradient = loss_function_derivative(y, output)
                # taking the layers in reversed order, so the gradient is propagated from the output layer to the input layer
                for layer in reversed(self.layers):
                    gradient = layer.backward_pass(gradient, learning_rate)

                # accuracy
                predicted_label = np.argmax(output)
                true_label = np.argmax(y)
                if predicted_label == true_label:
                    correct_predictions += 1

            average_error = error / len(x_train)
            accuracy = correct_predictions / len(x_train) * 100
            print(f"epoch: {i + 1}/{epochs},\terror = {average_error},\taccuracy = {accuracy}%")

In [36]:
import os
import cv2
import cupy as np

def load_and_preprocess_data(data_dir, image_size):
    images = []
    labels = []

    # loading images
    for filename in os.listdir(data_dir):
        # reading and "preprocessing" image
        img = cv2.imread(os.path.join(data_dir, filename))
        img = cv2.resize(img, (image_size, image_size))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        images.append(img)
        # deciding the label based on the file name (example: cat.1.png)
        label = filename.split('.')[0]
        labels.append(1 if label == "cat" else 0)

    # converting lists to np arrays
    images = np.array(images)
    labels = np.array(labels)

    # normalizing
    images = images.reshape(len(images), 1, image_size, image_size)
    images = images.astype("float32") / 255

    # shuffling the data
    permutation = np.random.permutation(len(images))
    images = images[permutation]
    labels = labels[permutation]

    return images, labels


image_size = 64
adjusted = image_size - 2   # image_size - kernel_size + 1

train_data_dir = 'E:/dogs_vs_cats/train/train_small'
test_data_dir = 'E:/dogs_vs_cats/train/train_small'

x_train, y_train = load_and_preprocess_data(train_data_dir, image_size)
x_test, y_test = load_and_preprocess_data(test_data_dir, image_size)

network = Network([
    Convolutional(input_shape=(1, image_size, image_size),
                  kernel_size=3,
                  depth=5),
    Sigmoid(),
    Reshape(input_shape=(5, adjusted, adjusted),
            output_shape=(5 * adjusted * adjusted, 1)),
    Dense(n_input=(5 * adjusted * adjusted),
          n_output=100),
    Sigmoid(),
    Dense(n_input=100,
          n_output=2),
    Sigmoid()
])

# Define the loss function
loss_fn = binary_cross_entropy
loss_prime_fn = binary_cross_entropy_derivative

print("TRAIN:")
# train
network.train(
    loss_function=binary_cross_entropy,
    loss_function_derivative=binary_cross_entropy_derivative,
    x_train=x_train,
    y_train=y_train,
    epochs=20,
    learning_rate=0.1
)

# test
print("TEST:")
for x, y in zip(x_test, y_test):
    output = network.predict(x)
    predicted = "cat" if np.argmax(output) == 1 else "dog"
    true = "cat" if np.argmax(y) == 1 else "dog"
    if predicted == true:
        print(f"pred: {predicted}, true: {true}")
    else:
        print(f"pred: {predicted}, true: {true}\tX")

epoch: 1/20,	error = 1.4148146332903442,	accuracy = 41.7910447761194%
epoch: 2/20,	error = 1.370079017181167,	accuracy = 29.601990049751244%
epoch: 3/20,	error = 1.3773581776157484,	accuracy = 35.82089552238806%
epoch: 4/20,	error = 1.3781877457310865,	accuracy = 34.82587064676617%
epoch: 5/20,	error = 1.3990815299189452,	accuracy = 64.17910447761194%
epoch: 6/20,	error = 1.40108821442568,	accuracy = 64.17910447761194%
epoch: 7/20,	error = 1.40112910186497,	accuracy = 56.71641791044776%
epoch: 8/20,	error = 1.3953581933402122,	accuracy = 57.2139303482587%
epoch: 9/20,	error = 1.4112008110707766,	accuracy = 42.78606965174129%
epoch: 10/20,	error = 1.4234228994370872,	accuracy = 38.80597014925373%
epoch: 11/20,	error = 1.4231597212963434,	accuracy = 38.80597014925373%
epoch: 12/20,	error = 1.4228963906791197,	accuracy = 39.55223880597015%
epoch: 13/20,	error = 1.422632976293077,	accuracy = 37.56218905472637%
epoch: 14/20,	error = 1.422369441157363,	accuracy = 39.30348258706468%
epoch: 15

In [40]:
image_size = 128
adjusted = image_size - 2   # image_size - kernel_size + 1
adjusted2 = adjusted - 2

network2 = Network([
    Convolutional(input_shape=(1, image_size, image_size),
                  kernel_size=3,
                  depth=10),
    Relu(),

    Convolutional(input_shape=(1, image_size, image_size),
                  kernel_size=3,
                  depth=20),
    Relu(),

    Reshape(input_shape=(5, adjusted2, adjusted2),
            output_shape=(5 * adjusted2 * adjusted2, 1)),

    Dense(n_input=(5 * adjusted2 * adjusted2),
          n_output=200),
    Sigmoid(),

    Dense(n_input=200,
          n_output=2),
    Sigmoid()
])

print("TRAIN:")
# train
network.train(
    loss_function=binary_cross_entropy,
    loss_function_derivative=binary_cross_entropy_derivative,
    x_train=x_train,
    y_train=y_train,
    epochs=40,
    learning_rate=0.1
)

TRAIN:
epoch: 1/40,	error = 1.4136103066316008,	accuracy = 37.06467661691542%
epoch: 2/40,	error = 1.4135960105401635,	accuracy = 37.06467661691542%
epoch: 3/40,	error = 1.413582551649084,	accuracy = 37.06467661691542%
epoch: 4/40,	error = 1.4135698731764563,	accuracy = 37.06467661691542%
epoch: 5/40,	error = 1.4135579224927413,	accuracy = 37.06467661691542%
epoch: 6/40,	error = 1.4135466508178496,	accuracy = 37.06467661691542%
epoch: 7/40,	error = 1.4135360129370302,	accuracy = 37.06467661691542%
epoch: 8/40,	error = 1.413525966935246,	accuracy = 37.06467661691542%
epoch: 9/40,	error = 1.4135164739494723,	accuracy = 37.06467661691542%
epoch: 10/40,	error = 1.4135074979382707,	accuracy = 37.06467661691542%
epoch: 11/40,	error = 1.413499005467803,	accuracy = 37.06467661691542%
epoch: 12/40,	error = 1.4134909655134806,	accuracy = 37.06467661691542%
epoch: 13/40,	error = 1.4134833492763426,	accuracy = 37.06467661691542%
epoch: 14/40,	error = 1.4134761300132603,	accuracy = 37.0646766169154